In [221]:
from IPython.display import HTML, display

def show(df, height=350):
    html = f'<div style="height:{height}px; overflow:auto; border:1px solid #ccc;">{df.to_html()}</div>'
    display(HTML(html))

In [222]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client
from datetime import date, datetime
from typing import Optional

load_dotenv('../.env')
print('SUPABASE_URL loaded:         ', bool(os.environ.get('SUPABASE_URL')))
print('SUPABASE_SERVICE_KEY loaded: ', bool(os.environ.get('SUPABASE_SERVICE_KEY')))

SUPABASE_URL loaded:          True
SUPABASE_SERVICE_KEY loaded:  True


## 1. Supabase Client

In [223]:
_client: Client | None = None

def _get_client() -> Client:
    global _client
    if _client is None:
        _client = create_client(
            os.environ['SUPABASE_URL'],
            os.environ['SUPABASE_SERVICE_KEY'],
        )
    return _client

client = _get_client()
print('Supabase client:', client)

Supabase client: <supabase._sync.client.Client object at 0x0000022E0E9A5650>


## 2. Date Helpers

`_parse_date` converts the raw date strings from bank PDFs (e.g. `'Oct 3'`, `'10/03'`) into a proper `date` object with the correct year attached.

In [224]:
_DATE_FORMATS = ['%b %d', '%B %d', '%m/%d', '%m/%d/%Y', '%m/%d/%y']

def _parse_date(date_str: str, year: int) -> Optional[date]:
    if not date_str or str(date_str).lower() in ('nan', 'none', ''):
        return None
    date_str = str(date_str).strip()
    for fmt in _DATE_FORMATS:
        try:
            parsed = datetime.strptime(date_str, fmt)
            return parsed.replace(year=year).date()
        except ValueError:
            continue
    return None

# Test cases
test_dates = [
    ('Oct 3',    2025),
    ('October 3', 2025),
    ('10/03',    2025),
    ('10/03/25', 2025),
    ('',         2025),   # empty -> None
    ('nan',      2025),   # NaN string -> None
    ('garbage',  2025),   # unparseable -> None
]
for raw, year in test_dates:
    print(f'{raw!r:15} year={year} -> {_parse_date(raw, year)}')

'Oct 3'         year=2025 -> 2025-10-03
'October 3'     year=2025 -> 2025-10-03
'10/03'         year=2025 -> 2025-10-03
'10/03/25'      year=2025 -> 2025-10-03
''              year=2025 -> None
'nan'           year=2025 -> None
'garbage'       year=2025 -> None


## 3. Amount / Type Helper

Bank CSVs use sign to indicate direction: negative = money coming in (credit/refund), positive = money going out (debit/purchase).

In [225]:
def _to_type(amount: float) -> str:
    return 'credit' if amount < 0 else 'debit'

# Test cases
for amount in [-7.89, -100.00, 0.00, 15.59, 1192.76]:
    print(f'{amount:>10.2f} -> {_to_type(amount)}')

     -7.89 -> credit
   -100.00 -> credit
      0.00 -> debit
     15.59 -> debit
   1192.76 -> debit


## 4. Account Helpers

`_get_or_create_account` looks up an account by `last_four` (or `account_name` if no last four). If it doesn't exist yet it inserts a new row and returns the new UUID. This means the same card is never duplicated across multiple statement uploads.

In [226]:
def _get_or_create_account(
    client: Client,
    user_id: str,
    account_name: str,
    account_type: str,
    last_four: Optional[str] = None,
) -> str:
    if last_four:
        res = (
            client.table('accounts')
            .select('bank_acc_id')
            .eq('user_id', user_id)
            .eq('last_four', last_four)
            .execute()
        )
    else:
        res = (
            client.table('accounts')
            .select('bank_acc_id')
            .eq('user_id', user_id)
            .eq('account_name', account_name)
            .execute()
        )

    if res.data:
        print(f'Account already exists: {res.data[0]["bank_acc_id"]}')
        return res.data[0]['bank_acc_id']

    res = client.table('accounts').insert({
        'user_id': user_id,
        'account_name': account_name,
        'account_type': account_type,
        'last_four': last_four,
    }).execute()
    print(f'Created new account: {res.data[0]["bank_acc_id"]}')
    return res.data[0]['bank_acc_id']

In [227]:
# Peek at what accounts already exist
res = client.table('accounts').select('*').execute()
pd.DataFrame(res.data)

,bank_acc_id,user_id,account_name,account_type,last_four,created_at
0,bb95a558-fd4e-43cb-b14b-cf72db6362b9,184cc8ea-2430-43d6-8111-1f6297096658,Chase College Checking ****8585,checking,8585,2026-05-02T19:08:53.081589+00:00
1,17cf0cd1-19aa-47f8-948d-48273b9d516e,184cc8ea-2430-43d6-8111-1f6297096658,Chase Sapphire Preferred ****1333,credit,1333,2026-05-02T19:29:28.598213+00:00
2,d97ef7ac-d27f-48bf-8bb3-58a806d24d23,184cc8ea-2430-43d6-8111-1f6297096658,Capital One VentureOne ****2952,credit,2952,2026-05-02T20:03:04.683935+00:00


## 5. Statement Helpers

`_statement_exists` checks by `file_hash` (SHA-256 of the PDF bytes) so the same file can never be uploaded twice. `_create_statement` inserts the statement record and returns its UUID.

In [228]:
def _statement_exists(client: Client, file_hash: str) -> Optional[str]:
    res = (
        client.table('statements')
        .select('statements_id')
        .eq('file_hash', file_hash)
        .execute()
    )
    return res.data[0]['statements_id'] if res.data else None

def _create_statement(
    client: Client,
    user_id: str,
    account_id: str,
    filename: str,
    file_hash: str,
    storage_path: str,
    period_start: Optional[date],
    period_end: Optional[date],
) -> str:
    res = client.table('statements').insert({
        'user_id': user_id,
        'account_id': account_id,
        'filename': filename,
        'file_hash': file_hash,
        'storage_path': storage_path,
        'period_start': period_start.isoformat() if period_start else None,
        'period_end': period_end.isoformat() if period_end else None,
    }).execute()
    return res.data[0]['statements_id']

# Peek at existing statements
res = client.table('statements').select('*').execute()
pd.DataFrame(res.data)

,statements_id,user_id,account_id,filename,file_hash,storage_path,period_start,period_end,uploaded_at
0,b21acde7-cf0d-49db-ab8f-ed167772f671,184cc8ea-2430-43d6-8111-1f6297096658,bb95a558-fd4e-43cb-b14b-cf72db6362b9,Chase_College_20260325-8585.pdf,51ed994a93f06b6baacc2b9d190f9f207be29ab2bfd570...,,2026-02-27,2026-03-25,2026-05-02T19:08:53.226267+00:00
1,009da3fe-83db-4c18-a0bb-a4e51e2b533a,184cc8ea-2430-43d6-8111-1f6297096658,17cf0cd1-19aa-47f8-948d-48273b9d516e,Chase_Sapphire_20251217-1333.pdf,5250795ecc6dd243f538f0e9e192ffb51dafb646401696...,,2025-11-18,2025-12-17,2026-05-02T19:29:28.738781+00:00
2,d6d6fd47-03a9-4cf4-9388-8e96431cbd90,184cc8ea-2430-43d6-8111-1f6297096658,d97ef7ac-d27f-48bf-8bb3-58a806d24d23,Capital_One_102025_2952.pdf,6e50adcf28f66bfaaee09f7403e57977c76b0b94f45dec...,,2025-09-20,2025-10-20,2026-05-02T20:03:04.883395+00:00


## 6. Merchant Category Cache

`get_cached_categories_bulk` does a single Supabase query for all descriptions at once and returns a `{description: category}` dict. `cache_categories_bulk` upserts new mappings in one call — if a description already exists it overwrites, otherwise it inserts.

In [229]:
def get_cached_categories_bulk(descriptions: list[str]) -> dict[str, str]:
    if not descriptions:
        return {}
    res = (
        client.table('merchant_categories')
        .select('description, category')
        .in_('description', descriptions)
        .execute()
    )
    return {row['description']: row['category'] for row in res.data}

def cache_categories_bulk(items: dict[str, str]) -> None:
    if not items:
        return
    client.table('merchant_categories').upsert([
        {'description': desc, 'category': cat}
        for desc, cat in items.items()
    ]).execute()

# Peek at the cache
res = client.table('merchant_categories').select('*').execute()
show(pd.DataFrame(res.data))

,description,category,created_at
0,Payment Thank You-Mobile,Personal,2026-05-02T19:02:36.954862+00:00
1,HM Hennes Mauritz UK L London,Shopping,2026-05-02T19:02:36.954862+00:00
2,TIAN TIAN MARKET - CANARY LONDON,Groceries,2026-05-02T19:02:36.954862+00:00
3,CL Chase Travel TRIPCHRG VA,Travel,2026-05-02T19:02:36.954862+00:00
4,VENICE TRANSPORT VENEZIA,Travel,2026-05-02T19:02:36.954862+00:00
5,TFL TRAVEL CH TFL.GOV.UK/CP,Travel,2026-05-02T19:02:36.954862+00:00
6,ASDA SUPERSTORE ISLE OF DOGS,Groceries,2026-05-02T19:02:36.954862+00:00
7,SAINSBURYS S/MKTS THE CITY -MAN,Groceries,2026-05-02T19:02:36.954862+00:00
8,SQ PEPPER STREET TAVERN London,Food & Drink,2026-05-02T19:02:36.954862+00:00
9,GlobalE /Goodai Global In New York NY,Shopping,2026-05-02T19:02:36.954862+00:00


In [230]:
# Test bulk lookup
test_descs = ['MCDONALDS LONDON', 'NETFLIX', 'TESCO SUPERSTORE', 'NOT IN CACHE', 'ASDA STORES LONDON']
result = get_cached_categories_bulk(test_descs)
for desc in test_descs:
    hit = result.get(desc, '<miss>')
    print(f'{desc!r:35} -> {hit}')

'MCDONALDS LONDON'                  -> <miss>
'NETFLIX'                           -> <miss>
'TESCO SUPERSTORE'                  -> <miss>
'NOT IN CACHE'                      -> <miss>
'ASDA STORES LONDON'                -> Groceries


## 7. `upload_transactions` — Full Pipeline

Orchestrates the full upload:
```
DataFrame
    -> get/create account
    -> dedup check by file_hash
    -> create statement record
    -> parse dates, build transaction rows
    -> bulk insert into transactions table
    -> return { statement_id, account_id, inserted, skipped }
```

In [231]:
def upload_transactions(
    df: pd.DataFrame,
    user_id: str,
    account_name: str,
    account_type: str,
    pdf_filename: str,
    pdf_hash: str,
    last_four: Optional[str] = None,
    storage_path: str = '',
    period_start: Optional[date] = None,
    period_end: Optional[date] = None,
    skip_if_exists: bool = True,
) -> dict:
    year = (period_end or period_start or date.today()).year

    account_id = _get_or_create_account(
        client, user_id, account_name, account_type, last_four
    )

    if skip_if_exists:
        existing_id = _statement_exists(client, pdf_hash)
        if existing_id:
            print(f'Already uploaded (statement {existing_id}), skipping.')
            return {'statement_id': existing_id, 'account_id': account_id, 'inserted': 0, 'skipped': True}

    statement_id = _create_statement(
        client, user_id, account_id, pdf_filename, pdf_hash, storage_path, period_start, period_end
    )

    rows = []
    for row in df.to_dict('records'):
        trans_date = _parse_date(row.get('trans_date'), year)
        if trans_date is None:
            continue
        amount_raw = float(row['amount1'])
        category = row.get('category')
        rows.append({
            'statement_id': statement_id,
            'user_id': user_id,
            'date': trans_date.isoformat(),
            'description': str(row['description']).strip(),
            'amount': abs(amount_raw),
            'type': _to_type(amount_raw),
            'category': category if pd.notna(category) and category != '' else None,
        })

    if rows:
        client.table('transactions').insert(rows).execute()

    print(f'Inserted {len(rows)} transactions into statement {statement_id}')
    return {'statement_id': statement_id, 'account_id': account_id, 'inserted': len(rows), 'skipped': False}

In [232]:
from pipeline.pdf_parser import parse_pdf

PDF_PATH = '../data/Capital_One_102025_2952.pdf'
# PDF_PATH = '../data/Chase_College_20260325-8585.pdf'
# PDF_PATH = "../data/Chase_Sapphire_20251217-1333.pdf"

parsed = parse_pdf(PDF_PATH)
transactions_df = parsed['transactions']

# period_end is a 'YYYY-MM-DD' string, so parse it first to get the year
period_end   = date.fromisoformat(parsed['period_end'])   if parsed['period_end']   else None
period_start = date.fromisoformat(parsed['period_start']) if parsed['period_start'] else None
year = (period_end or period_start or date.today()).year

print(f"Card: {parsed['card_name']}  ****{parsed['last_four']}")
print(f"Period: {period_start} -> {period_end}  (year={year})")
print(f"{len(transactions_df)} transactions\n")

# Preview the rows that would be inserted — without touching Supabase
preview_rows = []
for row in transactions_df.to_dict('records'):
    trans_date = _parse_date(row.get('trans_date'), year)
    if trans_date is None:
        continue
    amount_raw = float(row['amount1'])
    preview_rows.append({
        'date':        trans_date.isoformat(),
        'description': str(row['description']).strip(),
        'amount':      abs(amount_raw),
        'type':        _to_type(amount_raw),
    })

print(f'{len(preview_rows)} rows ready to insert')
show(pd.DataFrame(preview_rows))

Card: Capital One VentureOne  ****2952
Period: 2025-09-20 -> 2025-10-20  (year=2025)
73 transactions

73 rows ready to insert


,date,description,amount,type
0,2025-10-15,aliexpressSan MateoCA -,7.89,credit
1,2025-10-17,GETYOURGUIDE TICKETSLONDONGBR -,109.79,credit
2,2025-10-17,GETYOURGUIDE TICKETSLONDONGBR -,34.41,credit
3,2025-09-19,WWW.VOXI.CO.UKVODAFONE LTD,13.73,debit
4,2025-09-21,SumUp *fresh meetcanning townGBR,15.59,debit
5,2025-09-21,TFL TRAVEL CHTFL.GOV.UK/CP,5.43,debit
6,2025-09-21,CGFLTCGLondonGBR,14.92,debit
7,2025-09-22,Zettle_*DOUBLE SEVEN HLondonGBR,15.53,debit
8,2025-09-22,TFL TRAVEL CHTFL.GOV.UK/CP,8.68,debit
9,2025-09-22,LOON FUNGLONDON W1D,38.80,debit


In [233]:
import hashlib

# Your Supabase auth UUID — find it in Supabase dashboard > Authentication > Users
USER_ID = '184cc8ea-2430-43d6-8111-1f6297096658'

pdf_bytes = open(PDF_PATH, 'rb').read()
pdf_hash  = hashlib.sha256(pdf_bytes).hexdigest()

result = upload_transactions(
    df           = transactions_df,
    user_id      = USER_ID,
    account_name = f"{parsed['card_name']} ****{parsed['last_four']}",
    account_type = parsed['account_type'] or 'credit',
    pdf_filename = os.path.basename(PDF_PATH),
    pdf_hash     = pdf_hash,
    last_four    = parsed['last_four'],
    period_start = period_start,
    period_end   = period_end,
    skip_if_exists = True,
)

print(result)

Account already exists: d97ef7ac-d27f-48bf-8bb3-58a806d24d23
Already uploaded (statement d6d6fd47-03a9-4cf4-9388-8e96431cbd90), skipping.
{'statement_id': 'd6d6fd47-03a9-4cf4-9388-8e96431cbd90', 'account_id': 'd97ef7ac-d27f-48bf-8bb3-58a806d24d23', 'inserted': 0, 'skipped': True}
